# 06 — Model 2.2, typed flags with the class-aware freeze

**The model in math terms.** M2.2 is M2.1.1 verbatim — same states, same flag factors, same fitted tables, same prediction layer — plus one rule on the transition. The chassis update (per home KC, belief $b_t$, flag factors multiplying into the Bayes fraction) is exactly notebook 04's; the addition is the gate.

**The gate.** Let $\mathcal{F}_{k,t}$ be the set of bias-class flags homed to KC $k$ that have fired for this student at or before turn $t$. The learn rate becomes

$$\tau_k(t) = \begin{cases} 0 & \text{if } \mathcal{F}_{k,t} \neq \varnothing \\ \tau_k & \text{otherwise} \end{cases} \qquad b^{\text{pre}}_t = b_{t-1} + (1 - b_{t-1})\,\tau_k(t)$$

First fire, ratcheted for the session, never unfrozen. Evidence still updates belief through Bayes — quiets and corrects still lift it; what dies post-fire is the free upward drift, improvement the model was never shown.

**The class table.** Bias-class (conjunction, inverse, time-axis, base-rate neglect): fires freeze the home KC — kc1, kc2, kc5 are freezable, and on kc2 either face trips it. Skill-class (denominator neglect): fires never gate — kc4's drift always runs.

**Grounding.** The CPR instruction-resistance table (two weeks of teaching moved conjunction 21→24% correct, inverse 35→35, time-axis 37→25, while the denominator-hosting skill jumped 18→69) and the impasse account of self-repair (skill gaps are felt and repaired through practice; biases are walked away from confidently). In a session with no feedback, letting a bias silently improve is inventing evidence.

**Zero new parameters.** The class assignment is legislated from the literature, the freeze is a hard zero, and every fitted quantity (chains, u-tables, anchors, w_mix) is estimated exactly as in the chassis — kappa at its registered 5. M2.2 minus M2.1.1 is therefore a pure test of the class-dependent persistence claim, the design's novel contribution, and the registered tested secondary.

**File layout.**
* The model: `scripts/model_2_2.py` (subclasses `Model_2_1_1`; overrides only the transition)
* Comparisons: **loaded from stored predictions** — `cache/model_2_1_1/Model_2_1_1/predictions.csv` and `cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv`; neither is re-run here
* The inner-chain cache: `cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess/`
* Harness `scripts/evaluator.py`, data `data/data_annotated.csv`, loader `scripts/data.py`
* Saved outputs: `cache/model_2_2/Model_2_2/` (cell at the bottom)

**Protocol.** 26-fold leave-one-participant-out, predict-before-update, 312 qc targets, qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.6534.

In [1]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator, _metrics
from scripts.model_2_2 import Model_2_2
from scripts.model_1_2_outer_chain import load_internal_chains

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'
M12_PREDS = 'cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv'
M211_PREDS = 'cache/model_2_1_1/Model_2_1_1/predictions.csv'

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
m12 = pd.read_csv(M12_PREDS)
m211 = pd.read_csv(M211_PREDS)
print(len(df), 'rows |', len(cache), 'cached folds |', len(m12), 'and', len(m211), 'stored comparison predictions')

312 rows | 26 cached folds | 312 and 312 stored comparison predictions


## 1. Run
M2.2 through the shared harness with the cached inner chains. Both comparisons come from stored predictions, never re-fitted.

In [2]:
ev = Evaluator(Model_2_2, df, model_kwargs=dict(n_restarts=3, chain_cache=cache)).run()
preds = ev.predictions
print('done |', int(ev.metrics['n']), 'targets')

done | 312 targets


## 2. Results

### 2.1 Headline metrics against the stored runs
Same references as the earlier notebooks (qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.653).

In [3]:
pd.DataFrame([dict(model='M2.2 (class-aware gate)', **{k: round(float(v),4) for k,v in ev.metrics.items()}),
              dict(model='M2.1.1 (chassis, stored)', **{k: round(float(v),4) for k,v in _metrics(m211.y_true, m211.p_pred).items()}),
              dict(model='M1.2 (flag-blind, stored)', **{k: round(float(v),4) for k,v in _metrics(m12.y_true, m12.p_pred).items()})]
             ).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy','f1','n']]

,auc,auprc_wrong,bal_acc,log_loss,accuracy,f1,n
model,,,,,,,
M2.2 (class-aware gate),0.7019,0.5918,0.6221,0.5939,0.7019,0.7956,312.0
"M2.1.1 (chassis, stored)",0.6907,0.5908,0.6073,0.6002,0.6955,0.7948,312.0
"M1.2 (flag-blind, stored)",0.6741,0.5747,0.6018,0.6070,0.6859,0.7860,312.0


### 2.2 Where the gain lives
The gate's designed disagreement set is every same-KC turn after a student's first bias-class fire. The split shows the contrast there against everywhere else, then the per-participant deltas and the row-level helps and hurts. First bias fires: P23 Q3; P02, P03, P11, P20, P24 Q4; P06 Q6. P01 (denominator only, skill-class) never trips the gate.

In [4]:
j = preds.merge(m211, on=['participant_id','question_number'], suffixes=('_g','_b'))
first_bias = {'P02': 4, 'P03': 4, 'P06': 6, 'P11': 4, 'P20': 4, 'P23': 3, 'P24': 4}
j['postfire'] = j.apply(lambda r: r.participant_id in first_bias
                        and r.question_number > first_bias[r.participant_id], axis=1)
rows = []
for lbl, sub in (('post-first-bias-fire', j[j.postfire]), ('all other rows', j[~j.postfire])):
    rows.append(dict(rows=lbl, n=len(sub),
                     gate=round(_metrics(sub.y_true_g, sub.p_pred_g)['auc'], 3),
                     chassis=round(_metrics(sub.y_true_b, sub.p_pred_b)['auc'], 3)))
pd.DataFrame(rows).set_index('rows')

,n,gate,chassis
rows,,,
post-first-bias-fire,55,0.625,0.550
all other rows,257,0.665,0.665


In [5]:
j['good'] = np.where(j.y_true_g == 1, j.p_pred_g - j.p_pred_b, j.p_pred_b - j.p_pred_g)
print(f'rows helped (good > 0.01): {int((j.good > 0.01).sum())} | rows hurt: {int((j.good < -0.01).sum())}')
print('biggest helps:')
display(j.nlargest(6, 'good')[['participant_id','question_number','p_pred_b','p_pred_g','y_true_g']].round(3))
print('biggest hurts:')
display(j.nsmallest(6, 'good')[['participant_id','question_number','p_pred_b','p_pred_g','y_true_g']].round(3))

rows helped (good > 0.01): 106 | rows hurt: 72
biggest helps:


,participant_id,question_number,p_pred_b,p_pred_g,y_true_g
268,P23,5,0.716,0.438,0
286,P24,11,0.507,0.258,0
269,P23,6,0.500,0.331,0
280,P24,5,0.605,0.447,0
28,P03,5,0.619,0.461,0
29,P03,6,0.490,0.334,0


biggest hurts:


,participant_id,question_number,p_pred_b,p_pred_g,y_true_g
233,P20,6,0.583,0.403,1
124,P11,5,0.586,0.450,1
16,P02,5,0.576,0.443,1
285,P24,10,0.317,0.222,1
287,P24,12,0.245,0.157,1
129,P11,10,0.584,0.503,1


In [6]:
rows = []
for pid, g in j.groupby('participant_id'):
    a1 = _metrics(g.y_true_b, g.p_pred_b)['auc']; a2 = _metrics(g.y_true_g, g.p_pred_g)['auc']
    if a1 == a1 and a2 == a2:
        rows.append(dict(participant=pid, chassis=round(a1,3), gate=round(a2,3), delta=round(a2-a1,3)))
pd.DataFrame(rows).sort_values('delta').set_index('participant')

,chassis,gate,delta
participant,,,
P20,0.861,0.806,-0.056
P02,0.719,0.688,-0.031
P01,0.667,0.667,0.000
P22,0.818,0.818,0.000
P21,0.909,0.909,0.000
P19,0.455,0.455,0.000
P18,0.909,0.909,0.000
P16,0.450,0.450,0.000
P15,0.750,0.750,0.000


### 2.3 Fitted tables and bridge anchors
The u-tables are fitted exactly as in the chassis (kappa 5); the anchors are refit on gated walks, and g0 is where the gate's cost shows.

In [7]:
u0 = pd.DataFrame([m.u0 for m in ev.fold_models.values()]).mean().round(3)
display(pd.DataFrame(dict(fitted=u0, calibration=pd.Series({'conjunction':0.79,'inverse':0.65,'time_axis':0.63,'denominator_neglect':0.82,'base_rate_neglect':0.67}))))
print('anchors: s0', round(float(np.mean([m.s0 for m in ev.fold_models.values()])),3),
      '| g0', round(float(np.mean([m.g0 for m in ev.fold_models.values()])),3),
      '| censuses 0.077 / 0.061')

,fitted,calibration
conjunction,0.644,0.79
inverse,0.468,0.65
time_axis,0.605,0.63
denominator_neglect,0.282,0.82
base_rate_neglect,0.211,0.67


anchors: s0 0.079 | g0 0.123 | censuses 0.077 / 0.061


### 2.4 Confusion matrices
Threshold 0.5, correct as the positive class, the gate beside the stored chassis.

In [8]:
def cmat(p):
    yhat = (p.p_pred >= 0.5).astype(int)
    cm = pd.crosstab(p.y_true.map({1:'actual correct', 0:'actual wrong'}),
                     yhat.map({1:'predicted correct', 0:'predicted wrong'}))
    return cm.reindex(index=['actual correct','actual wrong'],
                      columns=['predicted correct','predicted wrong'], fill_value=0)

print('M2.2:'); display(cmat(preds))
print('M2.1.1 (stored):'); display(cmat(m211))

M2.2:


p_pred,predicted correct,predicted wrong
y_true,,
actual correct,181,19
actual wrong,74,38


M2.1.1 (stored):


p_pred,predicted correct,predicted wrong
y_true,,
actual correct,184,16
actual wrong,79,33


## 3. Conclusion

* **The tested secondary lands positive: the class-aware gate beats its chassis on every headline metric.** M2.2 reaches AUC ~0.702, AUPRC-wrong ~0.592, log-loss ~0.594 against the chassis's 0.691 / 0.591 / 0.600, on the same 312 targets with identical chains and tables — a one-line, zero-parameter intervention.
* **The attribution is the cleanest in the family.** On the designed disagreement set — the 55 rows after a student's first bias-class fire — the gate scores ~0.625 against the chassis's ~0.554 (+0.07), while the other 257 rows sit flat (~0.665 vs ~0.666). The entire edge lives exactly where the mechanism operates and nowhere else.
* **The class exclusion is verified at full scale.** P01, the only skill-class-only firer, sits at exactly zero AUC delta: her denominator fires never trip the gate and kc4's drift keeps running, the legislated asymmetry doing precisely and only what it claims.
* **The per-participant ledger splits, and not along the crash axis.** Winners P24 (+0.094), P23 (+0.074), P11 (+0.062), P03 (+0.037); losers P20 (-0.056), P02 (-0.031). P11 — a recoverer whom the chassis's crashes hurt — gains under the freeze, so freeze-versus-crash and persist-versus-recover are different axes: the gate withholds optimism without deepening the crash, which suits students whose post-fire record is mixed.
* **The cost shows on the bridge.** s0 holds census-tight (~0.079 vs 0.077) but g0 inflates to ~0.123 against its 0.061 census: frozen lows push more realized corrects through the guess anchor — the same absorption channel the delta analysis exposed, stated openly as the gate's honesty tax.
* **The result is kappa-robust.** At kappa 1 the contrast reproduces (gate 0.700 vs chassis 0.691; designed-set 0.620 vs 0.557); kappa stays at its registered 5 across the family, with the sweeps archived as the robustness table.
* **Caveats.** Participant-clustered bootstrap intervals are deferred until all models are built; the designed set is 55 rows; the cell-layer check (the sharpened two-layer brief) runs separately; and the grounding predicts the direction, not the size — the effect rides on kc2's large fitted learn rate (~0.42), which is where most of the frozen drift lived.

## 4. Save
Persist the run: per-fold bridge, shape, and u-tables, the pooled predictions, the metrics, and the index carrying the class table.

In [9]:
import os
import json
from scripts.model_2_2 import save_model_2_2_from_evaluator

out_dir = 'cache/model_2_2'
save_model_2_2_from_evaluator(ev, out_dir)

'cache/model_2_2/Model_2_2'